In [ ]:
import copy
import os.path as osp

import numpy as np
import pandas as pd
from calisim.abc import (
	ApproximateBayesianComputationMethod,
	ApproximateBayesianComputationMethodModel,
)
from calisim.data_model import (
	DistributionModel,
	ParameterDataType,
	ParameterSpecification,
)
from calisim.statistics import CosineDistance
from pcse.base import ParameterProvider
from pcse.input import (
	DummySoilDataProvider,
	NASAPowerWeatherDataProvider,
	WOFOST72SiteDataProvider,
	YAMLAgroManagementReader,
	YAMLCropDataProvider,
)
from pcse.models import Wofost72_PP

In [ ]:
wdp = NASAPowerWeatherDataProvider(latitude=52, longitude=5)
print(wdp)

In [ ]:
sited = WOFOST72SiteDataProvider(WAV=50)
print(sited)

In [ ]:
soild = DummySoilDataProvider()
print(soild)

In [ ]:
cropd = YAMLCropDataProvider(fpath=osp.join("..", "data"), force_reload=True)
print(cropd)

In [ ]:
agro = YAMLAgroManagementReader(osp.join("..", "data", "AGMT_C2_2020.agro"))
print(agro)

In [ ]:
params = ParameterProvider(cropdata=cropd, sitedata=sited, soildata=soild)
wofost = Wofost72_PP(params, wdp, agro)
wofost.run_till_terminate()
observed_data = pd.DataFrame(wofost.get_output())
observed_data

In [ ]:
parameter_spec = ParameterSpecification(
	parameters=[
		DistributionModel(
			name="TDWI",
			distribution_name="truncnorm",
			distribution_args=[(20 - 75) / 10, (110 - 75) / 10],
			distribution_kwargs=dict(loc=75, scale=10),
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="SPAN",
			distribution_name="truncnorm",
			distribution_args=[(0 - 37) / 5, (60 - 37) / 5],
			distribution_kwargs=dict(loc=37, scale=5),
			data_type=ParameterDataType.CONTINUOUS,
		),
	]
)

In [ ]:
def abc_func(
	parameters: dict, simulation_id: str, observed_data: np.ndarray | None
) -> float | list[float]:
	p = copy.deepcopy(params)
	for k in parameters:
		p.set_override(k, parameters[k])

	wofost = Wofost72_PP(p, wdp, agro)
	wofost.run_till_terminate()
	simulated_data = pd.DataFrame(wofost.get_output()).LAI.describe().values

	metric = CosineDistance()
	discrepancy = metric.calculate(observed_data, simulated_data)
	return discrepancy

In [ ]:
specification = ApproximateBayesianComputationMethodModel(
	experiment_name="pyabc_approximate_bayesian_computation",
	parameter_spec=parameter_spec,
	observed_data=observed_data.LAI.describe().values,
	n_init=35,
	walltime=5,  # minutes
	epsilon=10,
	output_labels=["Leaf Area Index Discrepancy"],
	n_bootstrap=35,
	min_population_size=35,
	verbose=True,
	batched=False,
	random_seed=100,
	method_kwargs=dict(
		max_total_nr_simulations=250, max_nr_populations=50, min_acceptance_rate=0.0
	),
)

calibrator = ApproximateBayesianComputationMethod(
	calibration_func=abc_func, specification=specification, engine="pyabc"
)

calibrator.specify().execute().analyze()